# 00 - Messy School Data Generator

This notebook creates a set of **deliberately messy** tables in `catalog_40_copper_analyst_training.bronze` for use in the data modelling and standardisation exercises.

The data simulates termly snapshots of schools, pupils, enrolments, and assessments — each riddled with the data quality issues covered in notebooks 02–06:

* **Duplicates** — exact duplicates and natural key conflicts
* **Inconsistent labels** — case differences, whitespace, abbreviations
* **Schema drift** — column names differ between snapshots
* **Orphaned foreign keys** — references to schools/pupils that don’t exist
* **Null handling** — mix of `NULL`, empty strings, and placeholder values
* **Date format inconsistencies**
* **JSON fields** — nested metadata for data engineering exercises

Run all cells in order to (re)create the schema and tables.

In [0]:
-- Schools snapshot: Autumn 2024
-- Issues: inconsistent casing, whitespace, abbreviations, NULL vs empty string, duplicate URNs
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.schools_autumn_2024 AS
SELECT * FROM VALUES
  -- Clean records
  (100001, 'Oak Academy',           'London',       'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-06-15"}'),
  (100002, 'Elm Community School',  'Manchester',   'Maintained', 'Open',   'Primary',   '{"ofsted_rating": "Outstanding", "last_inspection": "2022-11-20"}'),
  (100003, 'Birch College',         'Birmingham',   'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Requires Improvement", "last_inspection": "2024-01-10"}'),
  (100004, 'Cedar Primary',         'Leeds',        'Maintained', 'Open',   'Primary',   '{"ofsted_rating": "Good", "last_inspection": "2023-09-05"}'),
  (100005, 'Willow Park School',    'Bristol',      'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-03-22"}'),
  -- Inconsistent casing and whitespace
  (100006, 'maple grove primary',   'SHEFFIELD',    'maintained', 'Open',   'primary',   '{"ofsted_rating": "Good", "last_inspection": "2023-07-01"}'),
  (100007, ' Ash Valley School ',   'liverpool',    'Academy',    'open',   'Secondary', '{"ofsted_rating": "Inadequate", "last_inspection": "2024-02-28"}'),
  (100008, 'Pine Hills Acad.',      'London',       'Acad',       'Open',   'Sec',       '{"ofsted_rating": "Good", "last_inspection": "2023-11-15"}'),
  -- NULL and empty string inconsistencies
  (100009, 'Rowan Free School',     'Nottingham',   'Free School','Open',   '',          '{"ofsted_rating": null, "last_inspection": null}'),
  (100010, 'Hazel Grammar',         '',             'Academy',    'Open',   NULL,        '{"ofsted_rating": "Good"}'),
  (100011, 'Ivy Technical College', NULL,           'N/A',        'Unknown','Secondary', NULL),
  -- Exact duplicate
  (100001, 'Oak Academy',           'London',       'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-06-15"}'),
  -- Same URN, different data (natural key duplicate)
  (100005, 'Willow Park Academy',   'Bristol',      'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Outstanding", "last_inspection": "2024-03-01"}'),
  -- Additional schools for volume
  (100012, 'Sycamore High',         'Newcastle',    'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-05-20"}'),
  (100013, 'Poplar Lane Primary',   'Oxford',       'Maintained', 'Open',   'Primary',   '{"ofsted_rating": "Outstanding", "last_inspection": "2022-09-14"}'),
  (100014, 'Yew Tree School',       'Cambridge',    'Academy',    'Open',   'All-through','{"ofsted_rating": "Good", "last_inspection": "2023-08-30"}'),
  (100015, 'Laurel Park Academy',   'Bath',         'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Requires Improvement", "last_inspection": "2024-01-22"}'),
  (100016, 'Holly Cross School',    'York',         'Maintained', 'Open',   'Primary',   '{"ofsted_rating": "Good", "last_inspection": "2023-04-11"}'),
  (100017, 'Juniper Academy',       'Plymouth',     'Academy',    'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-12-05"}'),
  (100018, 'Alder Community School','Coventry',     'Maintained', 'Open',   'Primary',   '{"ofsted_rating": "Inadequate", "last_inspection": "2024-02-15"}')
AS t(school_urn, school_name, city, school_type, status, phase, metadata_json);

In [0]:
-- Schools snapshot: Spring 2025
-- Issues: DIFFERENT COLUMN NAMES (schema drift), more inconsistencies, a closed school, orphan-inducing gaps
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.schools_spring_2025 AS
SELECT * FROM VALUES
  -- Note: column names differ from autumn! 'establishment_name' not 'school_name', 'local_authority' not 'city', 'phase_of_education' not 'phase'
  (100001, 'Oak Academy',           'London',       'Academy',     'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-06-15", "pupil_premium_pct": 0.32}'),
  (100002, 'Elm Community School',  'Manchester',   'Maintained',  'Open',   'Primary',   '{"ofsted_rating": "Outstanding", "last_inspection": "2022-11-20", "pupil_premium_pct": 0.18}'),
  (100003, 'Birch College',         'Birmingham',   'Academy',     'Closed', 'Secondary', '{"ofsted_rating": "Requires Improvement", "last_inspection": "2024-01-10", "pupil_premium_pct": 0.45}'),
  (100004, 'Cedar Primary School',  'Leeds',        'Maintained',  'Open',   'Primary',   '{"ofsted_rating": "Good", "last_inspection": "2023-09-05", "pupil_premium_pct": 0.27}'),
  (100005, 'Willow Park School',    'Bristol',      'Academy',     'Open',   'Secondary', '{"ofsted_rating": "Outstanding", "last_inspection": "2024-03-01", "pupil_premium_pct": 0.21}'),
  (100006, 'Maple Grove Primary',   'Sheffield',    'Maintained',  'Open',   'Primary',   '{"ofsted_rating": "Good", "last_inspection": "2023-07-01", "pupil_premium_pct": 0.38}'),
  (100007, 'Ash Valley School',     'Liverpool',    'Academy',     'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2024-09-10", "pupil_premium_pct": 0.29}'),
  (100008, 'Pine Hills Academy',    'London',       'Academy',     'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-11-15", "pupil_premium_pct": 0.15}'),
  (100009, 'Rowan Free School',     'Nottingham',   'Free School', 'Open',   'All-through','{"ofsted_rating": "Good", "last_inspection": "2024-07-20", "pupil_premium_pct": 0.22}'),
  (100010, 'Hazel Grammar School',  'Canterbury',   'Academy',     'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2024-05-12", "pupil_premium_pct": 0.09}'),
  -- 100011 Ivy Technical College is MISSING from this snapshot entirely
  (100012, 'sycamore high',         'newcastle',    'academy',     'Open',   'secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-05-20", "pupil_premium_pct": 0.33}'),
  (100013, 'Poplar Lane Primary',   'Oxford',       'Maintained',  'Open',   'Primary',   '{"ofsted_rating": "Outstanding", "last_inspection": "2022-09-14", "pupil_premium_pct": 0.12}'),
  (100014, 'Yew Tree School',       'Cambridge',    'Academy',     'Open',   'All-through','{"ofsted_rating": "Good", "last_inspection": "2023-08-30", "pupil_premium_pct": 0.20}'),
  (100015, 'Laurel Park Academy',   'Bath',         'Academy',     'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2024-11-05", "pupil_premium_pct": 0.41}'),
  (100016, 'Holly Cross School',    'York',         'Maintained',  'Open',   'Primary',   '{"ofsted_rating": "Good", "last_inspection": "2023-04-11", "pupil_premium_pct": 0.35}'),
  (100017, 'Juniper Academy',       'Plymouth',     'Academy',     'Open',   'Secondary', '{"ofsted_rating": "Good", "last_inspection": "2023-12-05", "pupil_premium_pct": 0.24}'),
  (100018, 'Alder Community School','Coventry',     'Maintained',  'Open',   'Primary',   '{"ofsted_rating": "Good", "last_inspection": "2024-08-15", "pupil_premium_pct": 0.48}'),
  -- New school not in autumn snapshot
  (100019, 'Chestnut Academy',      'Reading',      'Academy',     'Open',   'Secondary', '{"ofsted_rating": null, "last_inspection": null, "pupil_premium_pct": null}')
AS t(urn, establishment_name, local_authority, establishment_type, status, phase_of_education, metadata_json);

In [0]:
-- Schools snapshot: Summer 2025
-- Issues: NEW COLUMN 'region' added, inconsistent region casing, NULL region values
-- Retains spring column names (urn, establishment_name, local_authority, etc.)
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.schools_summer_2025 AS
SELECT * FROM VALUES
  (100001, 'Oak Academy',           'London',       'Academy',     'Open',   'Secondary',   'London',          '{"ofsted_rating": "Good", "last_inspection": "2025-05-12", "pupil_premium_pct": 0.30}'),
  (100002, 'Elm Community School',  'Manchester',   'Maintained',  'Open',   'Primary',     'North West',      '{"ofsted_rating": "Outstanding", "last_inspection": "2022-11-20", "pupil_premium_pct": 0.18}'),
  (100003, 'Birch College',         'Birmingham',   'Academy',     'Closed', 'Secondary',   'West Midlands',   '{"ofsted_rating": "Requires Improvement", "last_inspection": "2024-01-10", "pupil_premium_pct": 0.45}'),
  (100004, 'Cedar Primary School',  'Leeds',        'Maintained',  'Open',   'Primary',     'Yorkshire',       '{"ofsted_rating": "Good", "last_inspection": "2023-09-05", "pupil_premium_pct": 0.27}'),
  (100005, 'Willow Park School',    'Bristol',      'Academy',     'Open',   'Secondary',   'South West',      '{"ofsted_rating": "Outstanding", "last_inspection": "2024-03-01", "pupil_premium_pct": 0.21}'),
  (100006, 'Maple Grove Primary',   'Sheffield',    'Maintained',  'Open',   'Primary',     'Yorkshire',       '{"ofsted_rating": "Good", "last_inspection": "2023-07-01", "pupil_premium_pct": 0.38}'),
  (100007, 'Ash Valley School',     'Liverpool',    'Academy',     'Open',   'Secondary',   'North West',      '{"ofsted_rating": "Good", "last_inspection": "2024-09-10", "pupil_premium_pct": 0.29}'),
  (100008, 'Pine Hills Academy',    'London',       'Academy',     'Open',   'Secondary',   'London',          '{"ofsted_rating": "Good", "last_inspection": "2023-11-15", "pupil_premium_pct": 0.15}'),
  (100009, 'Rowan Free School',     'Nottingham',   'Free School', 'Open',   'All-through', 'East Midlands',   '{"ofsted_rating": "Good", "last_inspection": "2024-07-20", "pupil_premium_pct": 0.22}'),
  (100010, 'Hazel Grammar School',  'Canterbury',   'Academy',     'Open',   'Secondary',   'South East',      '{"ofsted_rating": "Good", "last_inspection": "2024-05-12", "pupil_premium_pct": 0.09}'),
  -- Inconsistent region casing
  (100012, 'Sycamore High',         'Newcastle',    'Academy',     'Open',   'Secondary',   'north east',      '{"ofsted_rating": "Good", "last_inspection": "2023-05-20", "pupil_premium_pct": 0.33}'),
  (100013, 'Poplar Lane Primary',   'Oxford',       'Maintained',  'Open',   'Primary',     'South East',      '{"ofsted_rating": "Outstanding", "last_inspection": "2022-09-14", "pupil_premium_pct": 0.12}'),
  (100014, 'Yew Tree School',       'Cambridge',    'Academy',     'Open',   'All-through', 'east of england', '{"ofsted_rating": "Good", "last_inspection": "2023-08-30", "pupil_premium_pct": 0.20}'),
  (100015, 'Laurel Park Academy',   'Bath',         'Academy',     'Open',   'Secondary',   'South West',      '{"ofsted_rating": "Good", "last_inspection": "2024-11-05", "pupil_premium_pct": 0.41}'),
  (100016, 'Holly Cross School',    'York',         'Maintained',  'Open',   'Primary',     'Yorkshire',       '{"ofsted_rating": "Good", "last_inspection": "2023-04-11", "pupil_premium_pct": 0.35}'),
  (100017, 'Juniper Academy',       'Plymouth',     'Academy',     'Open',   'Secondary',   'South West',      '{"ofsted_rating": "Good", "last_inspection": "2023-12-05", "pupil_premium_pct": 0.24}'),
  (100018, 'Alder Community School','Coventry',     'Maintained',  'Open',   'Primary',     'West Midlands',   '{"ofsted_rating": "Good", "last_inspection": "2025-03-20", "pupil_premium_pct": 0.44}'),
  (100019, 'Chestnut Academy',      'Reading',      'Academy',     'Open',   'Secondary',   'South East',      '{"ofsted_rating": "Good", "last_inspection": "2025-06-01", "pupil_premium_pct": 0.17}'),
  -- NULL and empty region values
  (100020, 'Beech Park School',     'Brighton',     'Free School', 'Open',   'Primary',     NULL,              '{"ofsted_rating": null, "last_inspection": null, "pupil_premium_pct": null}'),
  (100021, 'Larch Gate Academy',    'Exeter',       'Academy',     'Open',   'Secondary',   '',                '{"ofsted_rating": null, "last_inspection": null, "pupil_premium_pct": null}')
AS t(urn, establishment_name, local_authority, establishment_type, status, phase_of_education, region, metadata_json);

In [0]:
-- Schools snapshot: Autumn 2025
-- Issues: COLUMN REMOVED — phase/phase_of_education is no longer in the extract
-- Schema drifts back to autumn-style column names (school_urn, school_name, city, etc.)
-- 100003 (Birch College) removed entirely after closure, 100011 still missing
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.schools_autumn_2025 AS
SELECT * FROM VALUES
  (100001, 'Oak Academy',           'London',       'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2025-05-12", "pupil_premium_pct": 0.30}'),
  (100002, 'Elm Community School',  'Manchester',   'Maintained',  'Open',   '{"ofsted_rating": "Outstanding", "last_inspection": "2025-09-15", "pupil_premium_pct": 0.16}'),
  (100004, 'Cedar Primary',         'Leeds',        'Maintained',  'Open',   '{"ofsted_rating": "Good", "last_inspection": "2023-09-05", "pupil_premium_pct": 0.27}'),
  (100005, 'Willow Park School',    'Bristol',      'Academy',     'Open',   '{"ofsted_rating": "Outstanding", "last_inspection": "2024-03-01", "pupil_premium_pct": 0.19}'),
  (100006, 'Maple Grove Primary',   'Sheffield',    'Maintained',  'Open',   '{"ofsted_rating": "Good", "last_inspection": "2023-07-01", "pupil_premium_pct": 0.38}'),
  (100007, 'Ash Valley School',     'Liverpool',    'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2024-09-10", "pupil_premium_pct": 0.29}'),
  (100008, 'Pine Hills Academy',    'London',       'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2023-11-15", "pupil_premium_pct": 0.15}'),
  (100009, 'Rowan Free School',     'Nottingham',   'Free School', 'Open',   '{"ofsted_rating": "Good", "last_inspection": "2024-07-20", "pupil_premium_pct": 0.22}'),
  (100010, 'Hazel Grammar',         'Canterbury',   'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2024-05-12", "pupil_premium_pct": 0.09}'),
  -- Inconsistent casing
  (100012, 'sycamore high',         'NEWCASTLE',    'academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2025-07-14", "pupil_premium_pct": 0.31}'),
  (100013, 'Poplar Lane Primary',   'Oxford',       'Maintained',  'Open',   '{"ofsted_rating": "Outstanding", "last_inspection": "2022-09-14", "pupil_premium_pct": 0.12}'),
  (100014, 'Yew Tree School',       'Cambridge',    'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2023-08-30", "pupil_premium_pct": 0.20}'),
  (100015, 'Laurel Park Academy',   'Bath',         'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2024-11-05", "pupil_premium_pct": 0.39}'),
  (100016, 'Holly Cross School',    'York',         'Maintained',  'Open',   '{"ofsted_rating": "Good", "last_inspection": "2023-04-11", "pupil_premium_pct": 0.35}'),
  (100017, 'Juniper Academy',       'Plymouth',     'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2023-12-05", "pupil_premium_pct": 0.24}'),
  (100018, 'Alder Community School','Coventry',     'Maintained',  'Open',   '{"ofsted_rating": "Good", "last_inspection": "2025-03-20", "pupil_premium_pct": 0.44}'),
  (100019, 'Chestnut Academy',      'Reading',      'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2025-06-01", "pupil_premium_pct": 0.17}'),
  (100020, 'Beech Park School',     'Brighton',     'Free School', 'Open',   '{"ofsted_rating": "Requires Improvement", "last_inspection": "2025-10-08", "pupil_premium_pct": 0.28}'),
  (100021, 'Larch Gate Academy',    'Exeter',       'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2025-11-01", "pupil_premium_pct": 0.22}'),
  -- Exact duplicate
  (100001, 'Oak Academy',           'London',       'Academy',     'Open',   '{"ofsted_rating": "Good", "last_inspection": "2025-05-12", "pupil_premium_pct": 0.30}')
AS t(school_urn, school_name, city, school_type, status, metadata_json);

In [0]:
-- Pupils snapshot: Autumn 2024
-- Issues: duplicates, inconsistent names, mixed date formats, orphaned school refs, JSON with nested contact info
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.pupils_autumn_2024 AS

WITH base_pupils AS (
  SELECT * FROM VALUES
    -- Clean records at various schools
    ('P001', 'Alice',   'Johnson',   'F', '2012-03-15', 100001, 'Year 7',  '{"contact": {"parent_name": "Sarah Johnson", "phone": "07700900001", "email": "s.johnson@email.com"}, "sen_status": "None", "fsm_eligible": false}'),
    ('P002', 'Bob',     'Smith',     'M', '2011-07-22', 100001, 'Year 8',  '{"contact": {"parent_name": "David Smith", "phone": "07700900002", "email": "d.smith@email.com"}, "sen_status": "SEN Support", "fsm_eligible": true}'),
    ('P003', 'Carol',   'Williams',  'F', '2012-11-01', 100002, 'Year 7',  '{"contact": {"parent_name": "Jane Williams", "phone": "07700900003", "email": "j.williams@email.com"}, "sen_status": "None", "fsm_eligible": false}'),
    ('P004', 'Dan',     'Brown',     'M', '2010-09-30', 100003, 'Year 9',  '{"contact": {"parent_name": "Mike Brown", "phone": "07700900004"}, "sen_status": "EHCP", "fsm_eligible": true}'),
    ('P005', 'Eve',     'Taylor',    'F', '2013-01-18', 100004, 'Year 6',  '{"contact": {"parent_name": "Lisa Taylor", "phone": "07700900005", "email": "l.taylor@email.com"}, "sen_status": "None", "fsm_eligible": false}'),
    ('P006', 'Frank',   'Davies',    'M', '2011-05-25', 100005, 'Year 8',  '{"contact": {"parent_name": "Helen Davies", "phone": "07700900006"}, "sen_status": "SEN Support", "fsm_eligible": false}'),
    ('P007', 'Grace',   'Evans',     'F', '2012-08-12', 100006, 'Year 7',  '{"contact": {"parent_name": "Tom Evans", "phone": "07700900007", "email": "t.evans@email.com"}, "sen_status": "None", "fsm_eligible": true}'),
    ('P008', 'Harry',   'Wilson',    'M', '2010-12-03', 100007, 'Year 9',  '{"contact": {"parent_name": "Claire Wilson"}, "sen_status": "EHCP", "fsm_eligible": true}'),
    ('P009', 'Isla',    'Thomas',    'F', '2013-04-07', 100008, 'Year 6',  '{"contact": {"parent_name": "Paul Thomas", "phone": "07700900009", "email": "p.thomas@email.com"}, "sen_status": "None", "fsm_eligible": false}'),
    ('P010', 'Jack',    'Roberts',   'M', '2011-10-19', 100009, 'Year 8',  '{"contact": {"parent_name": "Karen Roberts", "phone": "07700900010"}, "sen_status": "SEN Support", "fsm_eligible": true}'),
    -- Inconsistent casing and name variants
    ('P011', 'katie',   'JONES',     'F', '2012-06-28', 100010, 'Year 7',  '{"contact": {"parent_name": "Emma Jones", "phone": "07700900011"}, "sen_status": "None", "fsm_eligible": false}'),
    ('P012', 'LIAM',    'garcia',    'M', '15/02/2011', 100012, 'Yr 8',    '{"contact": {"parent_name": "Maria Garcia"}, "sen_status": "none", "fsm_eligible": false}'),
    ('P013', 'Mia',     'Anderson',  'F', '03-22-2013', 100013, 'year 6',  '{"contact": {"parent_name": "Chris Anderson", "phone": "07700900013", "email": "c.anderson@email.com"}, "sen_status": "None", "fsm_eligible": true}'),
    ('P014', 'Noah',    'Martinez',  'M', '2010-11-14', 100014, 'Year 9',  '{"contact": {"parent_name": "Ana Martinez", "phone": "07700900014"}, "sen_status": "SEN Support", "fsm_eligible": false}'),
    ('P015', 'Olivia',  'Lee',       'F', '2012-02-09', 100015, 'Year 7',  '{"contact": {"parent_name": "James Lee", "phone": "07700900015", "email": "j.lee@email.com"}, "sen_status": "None", "fsm_eligible": false}'),
    ('P016', 'Peter',   'Clark',     'M', '2011-08-17', 100016, 'Year 8',  '{"contact": {"parent_name": "Susan Clark", "phone": "07700900016"}, "sen_status": "None", "fsm_eligible": true}'),
    ('P017', 'Quinn',   'Lewis',     'F', '2013-07-03', 100017, 'Year 6',  '{"contact": {"parent_name": "Robert Lewis", "phone": "07700900017", "email": "r.lewis@email.com"}, "sen_status": "EHCP", "fsm_eligible": true}'),
    ('P018', 'Ryan',    'Walker',    'M', '2010-04-26', 100018, 'Year 9',  '{"contact": {"parent_name": "Michelle Walker"}, "sen_status": "None", "fsm_eligible": false}'),
    -- Orphaned foreign key: school 999999 doesn't exist
    ('P019', 'Sophie',  'Hall',      'F', '2012-09-11', 999999, 'Year 7',  '{"contact": {"parent_name": "Mark Hall", "phone": "07700900019"}, "sen_status": "None", "fsm_eligible": false}'),
    -- NULL and empty gender/year group
    ('P020', 'Tom',     'Wright',    'M', '2011-03-08', 100001, '',        '{"contact": {"parent_name": "Linda Wright"}, "sen_status": "Unknown", "fsm_eligible": null}'),
    ('P021', 'Uma',     'Young',     NULL,'2013-12-25', 100002, 'Year 6',  '{"contact": {}, "sen_status": null, "fsm_eligible": false}'),
    ('P022', 'Victor',  'King',      'M', NULL,         100005, 'Year 8',  '{"contact": {"parent_name": "Janet King", "phone": "07700900022"}, "sen_status": "N/A", "fsm_eligible": false}'),
    -- Exact duplicates
    ('P001', 'Alice',   'Johnson',   'F', '2012-03-15', 100001, 'Year 7',  '{"contact": {"parent_name": "Sarah Johnson", "phone": "07700900001", "email": "s.johnson@email.com"}, "sen_status": "None", "fsm_eligible": false}'),
    ('P006', 'Frank',   'Davies',    'M', '2011-05-25', 100005, 'Year 8',  '{"contact": {"parent_name": "Helen Davies", "phone": "07700900006"}, "sen_status": "SEN Support", "fsm_eligible": false}'),
    -- Natural key duplicate: same pupil_id, different data
    ('P002', 'Robert',  'Smith',     'M', '2011-07-22', 100001, 'Year 8',  '{"contact": {"parent_name": "David Smith", "phone": "07700900002", "email": "bob.smith@newemail.com"}, "sen_status": "SEN Support", "fsm_eligible": true}')
  AS t(pupil_id, first_name, last_name, gender, date_of_birth, school_urn, year_group, metadata_json)
)
SELECT * FROM base_pupils;

In [0]:
-- Pupils snapshot: Spring 2025
-- Issues: SCHEMA DRIFT (different column names), more orphans, moved pupils, inconsistencies
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.pupils_spring_2025 AS
SELECT * FROM VALUES
  -- Note: 'forename'/'surname' instead of 'first_name'/'last_name', 'dob' instead of 'date_of_birth', 'urn' instead of 'school_urn'
  ('P001', 'Alice',   'Johnson',   'Female',  '2012-03-15', 100001, 'Year 7',  '{"contact": {"parent_name": "Sarah Johnson", "phone": "07700900001", "email": "s.johnson@email.com"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 96.2}'),
  ('P002', 'Bob',     'Smith',     'Male',    '2011-07-22', 100001, 'Year 8',  '{"contact": {"parent_name": "David Smith", "phone": "07700900002", "email": "d.smith@email.com"}, "sen_status": "SEN Support", "fsm_eligible": true, "attendance_pct": 88.5}'),
  ('P003', 'Carol',   'Williams',  'Female',  '2012-11-01', 100002, 'Year 7',  '{"contact": {"parent_name": "Jane Williams", "phone": "07700900003", "email": "j.williams@email.com"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 97.8}'),
  -- P004 moved from Birch College (now closed) to Willow Park
  ('P004', 'Dan',     'Brown',     'Male',    '2010-09-30', 100005, 'Year 9',  '{"contact": {"parent_name": "Mike Brown", "phone": "07700900004"}, "sen_status": "EHCP", "fsm_eligible": true, "attendance_pct": 72.1}'),
  ('P005', 'Eve',     'Taylor',    'Female',  '2013-01-18', 100004, 'Year 6',  '{"contact": {"parent_name": "Lisa Taylor", "phone": "07700900005", "email": "l.taylor@email.com"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 99.0}'),
  ('P006', 'Frank',   'Davies',    'Male',    '2011-05-25', 100005, 'Year 8',  '{"contact": {"parent_name": "Helen Davies", "phone": "07700900006"}, "sen_status": "SEN Support", "fsm_eligible": false, "attendance_pct": 91.3}'),
  ('P007', 'Grace',   'Evans',     'Female',  '2012-08-12', 100006, 'Year 7',  '{"contact": {"parent_name": "Tom Evans", "phone": "07700900007", "email": "t.evans@email.com"}, "sen_status": "None", "fsm_eligible": true, "attendance_pct": 94.6}'),
  ('P008', 'Harry',   'Wilson',    'Male',    '2010-12-03', 100007, 'Year 9',  '{"contact": {"parent_name": "Claire Wilson"}, "sen_status": "EHCP", "fsm_eligible": true, "attendance_pct": 68.9}'),
  ('P009', 'Isla',    'Thomas',    'Female',  '2013-04-07', 100008, 'Year 6',  '{"contact": {"parent_name": "Paul Thomas", "phone": "07700900009", "email": "p.thomas@email.com"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 95.4}'),
  ('P010', 'Jack',    'Roberts',   'Male',    '2011-10-19', 100009, 'Year 8',  '{"contact": {"parent_name": "Karen Roberts", "phone": "07700900010"}, "sen_status": "SEN Support", "fsm_eligible": true, "attendance_pct": 85.2}'),
  ('P011', 'Katie',   'Jones',     'Female',  '2012-06-28', 100010, 'Year 7',  '{"contact": {"parent_name": "Emma Jones", "phone": "07700900011"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 98.1}'),
  ('P012', 'Liam',    'Garcia',    'Male',    '2011-02-15', 100012, 'Year 8',  '{"contact": {"parent_name": "Maria Garcia"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 90.7}'),
  ('P013', 'Mia',     'Anderson',  'Female',  '2013-03-22', 100013, 'Year 6',  '{"contact": {"parent_name": "Chris Anderson", "phone": "07700900013", "email": "c.anderson@email.com"}, "sen_status": "None", "fsm_eligible": true, "attendance_pct": 93.5}'),
  ('P014', 'Noah',    'Martinez',  'Male',    '2010-11-14', 100014, 'Year 9',  '{"contact": {"parent_name": "Ana Martinez", "phone": "07700900014"}, "sen_status": "SEN Support", "fsm_eligible": false, "attendance_pct": 87.3}'),
  ('P015', 'Olivia',  'Lee',       'Female',  '2012-02-09', 100015, 'Year 7',  '{"contact": {"parent_name": "James Lee", "phone": "07700900015", "email": "j.lee@email.com"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 96.9}'),
  ('P016', 'Peter',   'Clark',     'Male',    '2011-08-17', 100016, 'Year 8',  '{"contact": {"parent_name": "Susan Clark", "phone": "07700900016"}, "sen_status": "None", "fsm_eligible": true, "attendance_pct": 92.0}'),
  ('P017', 'Quinn',   'Lewis',     'Female',  '2013-07-03', 100017, 'Year 6',  '{"contact": {"parent_name": "Robert Lewis", "phone": "07700900017", "email": "r.lewis@email.com"}, "sen_status": "EHCP", "fsm_eligible": true, "attendance_pct": 78.4}'),
  ('P018', 'Ryan',    'Walker',    'Male',    '2010-04-26', 100018, 'Year 9',  '{"contact": {"parent_name": "Michelle Walker"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 94.1}'),
  -- Orphaned: school 999999 still doesn't exist
  ('P019', 'Sophie',  'Hall',      'Female',  '2012-09-11', 999999, 'Year 7',  '{"contact": {"parent_name": "Mark Hall", "phone": "07700900019"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 95.0}'),
  ('P020', 'Tom',     'Wright',    'Male',    '2011-03-08', 100001, 'Year 8',  '{"contact": {"parent_name": "Linda Wright"}, "sen_status": "SEN Support", "fsm_eligible": true, "attendance_pct": 82.6}'),
  ('P021', 'Uma',     'Young',     'Female',  '2013-12-25', 100002, 'Year 6',  '{"contact": {"parent_name": "Raj Young", "phone": "07700900021"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 97.3}'),
  ('P022', 'Victor',  'King',      'Male',    '2011-06-14', 100005, 'Year 8',  '{"contact": {"parent_name": "Janet King", "phone": "07700900022"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 91.8}'),
  -- New pupil at new school
  ('P023', 'Wendy',   'Scott',     'Female',  '2012-10-05', 100019, 'Year 7',  '{"contact": {"parent_name": "Brian Scott", "phone": "07700900023", "email": "b.scott@email.com"}, "sen_status": "None", "fsm_eligible": false, "attendance_pct": 96.5}'),
  -- Orphaned: references closed school 100003
  ('P024', 'Xander',  'Green',     'Male',    '2010-08-20', 100003, 'Year 9',  '{"contact": {"parent_name": "Fiona Green"}, "sen_status": "SEN Support", "fsm_eligible": true, "attendance_pct": 74.2}'),
  -- Exact duplicate
  ('P010', 'Jack',    'Roberts',   'Male',    '2011-10-19', 100009, 'Year 8',  '{"contact": {"parent_name": "Karen Roberts", "phone": "07700900010"}, "sen_status": "SEN Support", "fsm_eligible": true, "attendance_pct": 85.2}')
AS t(pupil_id, forename, surname, sex, dob, urn, year_group, metadata_json);

In [0]:
-- Assessment results spanning both terms
-- Issues: mixed date formats, inconsistent subject names, duplicates, NULLs, JSON with variable structure
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.assessments AS
SELECT * FROM VALUES
  -- Autumn 2024 assessments - clean-ish
  (1,  'P001', 100001, 'Mathematics',  72, 'B',  '2024-10-15', 'Autumn 2024', '{"teacher": "Mr Adams", "moderated": true, "method": "written_exam"}'),
  (2,  'P001', 100001, 'English',      85, 'A',  '2024-10-15', 'Autumn 2024', '{"teacher": "Ms Baker", "moderated": true, "method": "coursework"}'),
  (3,  'P002', 100001, 'Mathematics',  58, 'C',  '2024-10-15', 'Autumn 2024', '{"teacher": "Mr Adams", "moderated": true, "method": "written_exam"}'),
  (4,  'P002', 100001, 'English',      62, 'C',  '2024-10-15', 'Autumn 2024', '{"teacher": "Ms Baker", "moderated": false, "method": "coursework"}'),
  (5,  'P003', 100002, 'Mathematics',  91, 'A',  '2024-10-16', 'Autumn 2024', '{"teacher": "Mrs Carter", "moderated": true, "method": "written_exam"}'),
  (6,  'P003', 100002, 'English',      88, 'A',  '2024-10-16', 'Autumn 2024', '{"teacher": "Mr Dean", "moderated": true, "method": "coursework"}'),
  (7,  'P004', 100003, 'Mathematics',  45, 'D',  '2024-10-17', 'Autumn 2024', '{"teacher": "Ms Ellis", "moderated": true, "method": "written_exam"}'),
  (8,  'P004', 100003, 'English',      51, 'C',  '2024-10-17', 'Autumn 2024', '{"teacher": "Mr Ford", "moderated": false}'),
  (9,  'P005', 100004, 'Maths',        78, 'B',  '15/10/2024', 'Autumn 2024', '{"teacher": "Mrs Grant", "moderated": true, "method": "written_exam"}'),
  (10, 'P005', 100004, 'english',      82, 'A',  '15/10/2024', 'Autumn 2024', '{"teacher": "Mr Hunt", "moderated": true}'),
  (11, 'P006', 100005, 'MATHEMATICS',  67, 'B',  '2024-10-18', 'Autumn 2024', '{"teacher": "Ms Irving", "moderated": true, "method": "written_exam"}'),
  (12, 'P006', 100005, 'Science',      71, 'B',  '2024-10-18', 'Autumn 2024', '{"teacher": "Mr Jones", "moderated": true, "method": "practical"}'),
  (13, 'P007', 100006, 'Mathematics',  83, 'A',  '2024-10-18', 'Autumn 2024', '{"teacher": "Mrs Kelly", "moderated": true, "method": "written_exam"}'),
  (14, 'P008', 100007, 'Mathematics',  39, 'E',  '2024-10-19', 'Autumn 2024', '{"teacher": "Mr Lane", "moderated": false, "method": "written_exam"}'),
  (15, 'P009', 100008, 'Maths',        76, 'B',  '19/10/2024', 'Autumn 2024', '{"teacher": "Ms Moore"}'),
  (16, 'P010', 100009, 'Mathematics',  55, 'C',  '2024-10-20', 'Autumn 2024', '{"teacher": "Mr Nash", "moderated": true, "method": "written_exam"}'),
  -- Exact duplicate
  (1,  'P001', 100001, 'Mathematics',  72, 'B',  '2024-10-15', 'Autumn 2024', '{"teacher": "Mr Adams", "moderated": true, "method": "written_exam"}'),
  -- Spring 2025 assessments
  (17, 'P001', 100001, 'Mathematics',  78, 'B',  '2025-02-10', 'Spring 2025', '{"teacher": "Mr Adams", "moderated": true, "method": "written_exam", "improvement": true}'),
  (18, 'P001', 100001, 'English',      87, 'A',  '2025-02-10', 'Spring 2025', '{"teacher": "Ms Baker", "moderated": true, "method": "coursework", "improvement": true}'),
  (19, 'P002', 100001, 'math',         61, 'C',  '10/02/2025', 'Spring 2025', '{"teacher": "Mr Adams", "moderated": true, "method": "written_exam"}'),
  (20, 'P002', 100001, 'English',      65, 'C',  '2025-02-10', 'Spring 2025', '{"teacher": "Ms Baker", "moderated": true, "method": "coursework"}'),
  (21, 'P003', 100002, 'Mathematics',  93, 'A*', '2025-02-11', 'Spring 2025', '{"teacher": "Mrs Carter", "moderated": true, "method": "written_exam", "improvement": true}'),
  (22, 'P003', 100002, 'English',      90, 'A',  '2025-02-11', 'Spring 2025', '{"teacher": "Mr Dean", "moderated": true, "method": "coursework", "improvement": true}'),
  -- P004 moved school but assessment still references old closed school
  (23, 'P004', 100003, 'Mathematics',  48, 'D',  '2025-02-12', 'Spring 2025', '{"teacher": "Ms Ellis", "moderated": false, "method": "written_exam"}'),
  (24, 'P005', 100004, 'Mathematics',  81, 'A',  '2025-02-12', 'Spring 2025', '{"teacher": "Mrs Grant", "moderated": true, "method": "written_exam", "improvement": true}'),
  (25, 'P006', 100005, 'mathematics',  70, 'B',  '2025-02-13', 'Spring 2025', '{"teacher": "Ms Irving", "moderated": true, "method": "written_exam"}'),
  (26, 'P007', 100006, 'Mathematics',  86, 'A',  '2025-02-13', 'Spring 2025', '{"teacher": "Mrs Kelly", "moderated": true, "method": "written_exam", "improvement": true}'),
  (27, 'P008', 100007, 'Mathematics',  42, 'D',  '2025-02-14', 'Spring 2025', '{"teacher": "Mr Lane", "moderated": true, "method": "written_exam"}'),
  (28, 'P009', 100008, 'Maths',        79, 'B',  '14/02/2025', 'Spring 2025', '{"teacher": "Ms Moore", "moderated": true}'),
  (29, 'P010', 100009, 'Mathematics',  59, 'C',  '2025-02-15', 'Spring 2025', '{"teacher": "Mr Nash", "moderated": true, "method": "written_exam"}'),
  -- Assessment for orphaned pupil
  (30, 'P019', 999999, 'Mathematics',  64, 'C',  '2025-02-15', 'Spring 2025', '{"teacher": "Unknown", "moderated": false}'),
  -- NULL score and grade
  (31, 'P011', 100010, 'Mathematics',  NULL, NULL,'2025-02-15', 'Spring 2025', '{"teacher": "Mr Owen", "moderated": false, "notes": "Absent - illness"}'),
  -- Duplicate with conflicting score (natural key: assessment_id)
  (24, 'P005', 100004, 'Mathematics',  79, 'B',  '2025-02-12', 'Spring 2025', '{"teacher": "Mrs Grant", "moderated": false, "method": "written_exam"}')
AS t(assessment_id, pupil_id, school_urn, subject, score, grade, assessment_date, term, metadata_json);

In [0]:
-- Attendance data: a single combined table with both terms
-- Issues: inconsistent status codes, mixed date formats, orphaned refs, NULLs, JSON session detail
CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.bronze.attendance AS
WITH date_range AS (
  -- Generate school days for Autumn term (Sep-Dec 2024) and Spring term (Jan-Mar 2025)
  SELECT explode(sequence(
    to_date('2024-09-02'), to_date('2025-03-28'), interval 1 day
  )) AS attendance_date
),
school_days AS (
  SELECT attendance_date
  FROM date_range
  WHERE dayofweek(attendance_date) BETWEEN 2 AND 6  -- Monday to Friday
    AND attendance_date NOT BETWEEN '2024-10-28' AND '2024-11-01' -- Half term
    AND attendance_date NOT BETWEEN '2024-12-23' AND '2025-01-03' -- Christmas
    AND attendance_date NOT BETWEEN '2025-02-17' AND '2025-02-21' -- Half term
),
pupil_list AS (
  SELECT * FROM VALUES
    ('P001'), ('P002'), ('P003'), ('P004'), ('P005'),
    ('P006'), ('P007'), ('P008'), ('P009'), ('P010'),
    ('P011'), ('P012'), ('P013'), ('P014'), ('P015'),
    ('P016'), ('P017'), ('P018'), ('P019'), ('P020')
  AS t(pupil_id)
),
cross_joined AS (
  SELECT p.pupil_id, s.attendance_date
  FROM pupil_list p
  CROSS JOIN school_days s
),
with_status AS (
  SELECT
    pupil_id,
    attendance_date,
    -- Generate realistic but messy attendance codes
    CASE
      -- Most pupils present most days
      WHEN abs(hash(concat(pupil_id, cast(attendance_date AS STRING)))) % 100 < 85 THEN
        CASE WHEN abs(hash(concat('am', pupil_id, cast(attendance_date AS STRING)))) % 20 = 0 THEN 'present'  -- inconsistent: lowercase
             WHEN abs(hash(concat('var', pupil_id, cast(attendance_date AS STRING)))) % 30 = 0 THEN 'P'       -- code variant
             ELSE 'Present'
        END
      WHEN abs(hash(concat(pupil_id, cast(attendance_date AS STRING)))) % 100 < 92 THEN
        CASE WHEN abs(hash(concat('ill', pupil_id, cast(attendance_date AS STRING)))) % 3 = 0 THEN 'Illness'
             WHEN abs(hash(concat('ill2', pupil_id, cast(attendance_date AS STRING)))) % 3 = 1 THEN 'ill'
             ELSE 'I'  -- code variant
        END
      WHEN abs(hash(concat(pupil_id, cast(attendance_date AS STRING)))) % 100 < 96 THEN 'Authorised Absence'
      WHEN abs(hash(concat(pupil_id, cast(attendance_date AS STRING)))) % 100 < 98 THEN
        CASE WHEN abs(hash(concat('unauth', pupil_id, cast(attendance_date AS STRING)))) % 2 = 0 THEN 'Unauthorised'
             ELSE 'U'  -- code variant
        END
      ELSE
        CASE WHEN abs(hash(concat('null', pupil_id, cast(attendance_date AS STRING)))) % 3 = 0 THEN NULL
             WHEN abs(hash(concat('null2', pupil_id, cast(attendance_date AS STRING)))) % 3 = 1 THEN ''
             ELSE 'N/A'
        END
    END AS attendance_status,
    CASE
      WHEN abs(hash(concat('fmt', pupil_id, cast(attendance_date AS STRING)))) % 5 = 0
        THEN date_format(attendance_date, 'dd/MM/yyyy')    -- UK format variant
      WHEN abs(hash(concat('fmt2', pupil_id, cast(attendance_date AS STRING)))) % 7 = 0
        THEN date_format(attendance_date, 'MM-dd-yyyy')    -- US format variant
      ELSE cast(attendance_date AS STRING)                   -- ISO format
    END AS date_recorded,
    CASE
      WHEN abs(hash(concat('sess', pupil_id, cast(attendance_date AS STRING)))) % 4 = 0
        THEN concat('{"am": "', 
          CASE WHEN abs(hash(concat('am_s', pupil_id, cast(attendance_date AS STRING)))) % 10 < 8 THEN 'Present' ELSE 'Absent' END,
          '", "pm": "',
          CASE WHEN abs(hash(concat('pm_s', pupil_id, cast(attendance_date AS STRING)))) % 10 < 9 THEN 'Present' ELSE 'Absent' END,
          '", "late_arrival": ', 
          CASE WHEN abs(hash(concat('late', pupil_id, cast(attendance_date AS STRING)))) % 8 = 0 THEN 'true' ELSE 'false' END,
          '}')
      ELSE NULL
    END AS session_json
  FROM cross_joined
)
SELECT * FROM with_status;

In [0]:
-- Verify what we've created
SELECT 'schools_autumn_2024' AS table_name, COUNT(*) AS row_count FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024
UNION ALL
SELECT 'schools_spring_2025', COUNT(*) FROM catalog_40_copper_analyst_training.bronze.schools_spring_2025
UNION ALL
SELECT 'schools_summer_2025', COUNT(*) FROM catalog_40_copper_analyst_training.bronze.schools_summer_2025
UNION ALL
SELECT 'schools_autumn_2025', COUNT(*) FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2025
UNION ALL
SELECT 'pupils_autumn_2024', COUNT(*) FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2024
UNION ALL
SELECT 'pupils_spring_2025', COUNT(*) FROM catalog_40_copper_analyst_training.bronze.pupils_spring_2025
UNION ALL
SELECT 'assessments', COUNT(*) FROM catalog_40_copper_analyst_training.bronze.assessments
UNION ALL
SELECT 'attendance', COUNT(*) FROM catalog_40_copper_analyst_training.bronze.attendance
ORDER BY table_name;

## Export tables to CSV

Write each table out to a CSV file in the `resources/bronze` folder of this repository so that the data can be used without requiring access to the catalog.

In [0]:
%python
import os

base_path = "/Workspace/Users/nicholas.treece@education.gov.uk/databricks_code_learn/resources/bronze"
os.makedirs(base_path, exist_ok=True)

tables = [
    "schools_autumn_2024",
    "schools_spring_2025",
    "schools_summer_2025",
    "schools_autumn_2025",
    "pupils_autumn_2024",
    "pupils_spring_2025",
    "assessments",
    "attendance",
]

for table_name in tables:
    df = spark.table(f"catalog_40_copper_analyst_training.bronze.{table_name}")
    pdf = df.toPandas()
    out_file = os.path.join(base_path, f"{table_name}.csv")
    pdf.to_csv(out_file, index=False)
    print(f"{table_name}: {len(pdf)} rows -> {out_file}")

print("\nAll tables exported.")